# USGS Water-Use Data (NWDC)

The USGS [National Water Availability Assessment Data Companion
(NWDC)](https://water.usgs.gov/nwaa-data/) serves national-scale, *modeled*
water-use estimates that underlie the USGS National Water Availability
Assessment. Estimates are produced on a **HUC12** (12-digit hydrologic unit)
grid and can be queried for any county, state, or hydrologic unit. They are
the modern replacement for the retired legacy NWIS water-use service.

`dataretrieval` exposes the service through a single function,
`wateruse.get_wateruse`, which returns a tidy `pandas.DataFrame` plus a
metadata object. Available **models** (categories) include:

| model | description |
| --- | --- |
| `wu-public-supply-wd` | public-supply withdrawals |
| `wu-public-supply-cu` | public-supply consumptive use |
| `wu-irrigation-wd` | irrigation withdrawals |
| `wu-irrigation-cu` | irrigation consumptive use |
| `wu-thermoelectric` | thermoelectric-power water use |

Each model exposes its own set of **variables**; see the
[NWDC data catalog](https://water.usgs.gov/nwaa-data/) for the full list.

## A motivating question

> *Where does Wisconsin's public water supply come from — groundwater or
> surface water — and when through the year is demand highest?*

Answering it is a typical water-use workflow: pull a state's monthly
withdrawals split by source, aggregate the HUC12 grid to a statewide time
series, and look at the seasonal cycle and the groundwater/surface-water
split.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd

from dataretrieval import wateruse

## 1. Retrieve the data

We request three variables for the public-supply-withdrawals model:
`pswdtot` (total), `pswdgw` (groundwater source), and `pswdsw` (surface-water
source), for every HUC12 in Wisconsin, monthly, for calendar year 2020.

Wisconsin spans more HUC12s than fit on a single page, so the service returns
the result across several pages — `get_wateruse` follows the pagination links
and stitches the pages into one frame for you.

In [ ]:
df, md = wateruse.get_wateruse(
    model="wu-public-supply-wd",
    variable=["pswdtot", "pswdgw", "pswdsw"],
    state="WI",
    start_date="2020-01",
    end_date="2020-12",
    time_resolution="monthly",
)

print(f"{len(df):,} rows across {df['huc12_id'].nunique():,} HUC12 watersheds")
df.head()

The frame is in **long (tidy) form**: one row per HUC12 and month. The
`huc12_id` column is kept as a string so leading zeros are preserved, and each
value column carries its unit — here `_mgd`, million gallons per day.

## 2. Aggregate to a statewide monthly series

Summing across every HUC12 gives the statewide withdrawal for each month.

In [ ]:
statewide = df.groupby("year_month")[["pswdtot_mgd", "pswdgw_mgd", "pswdsw_mgd"]].sum()
statewide.index = pd.to_datetime(statewide.index)
statewide

## 3. The seasonal demand cycle

Public-supply withdrawals peak in summer, when outdoor use (lawns, gardens,
recreation) adds to baseline indoor demand.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(statewide.index, statewide["pswdtot_mgd"], marker="o", color="#1f77b4")
ax.set_title("Wisconsin public-supply withdrawals, 2020")
ax.set_ylabel("Withdrawal (million gallons/day)")
ax.set_xlabel("Month")
ax.grid(True, alpha=0.3)
fig.tight_layout()

## 4. Groundwater vs. surface water

Because total withdrawal is the sum of its groundwater and surface-water
components, a stacked area shows how each source contributes through the year.

In [ ]:
gw_share = statewide["pswdgw_mgd"].sum() / statewide["pswdtot_mgd"].sum()
print(f"Groundwater supplies {gw_share:.0%} of Wisconsin's public water supply")

fig, ax = plt.subplots(figsize=(9, 4))
ax.stackplot(
    statewide.index,
    statewide["pswdgw_mgd"],
    statewide["pswdsw_mgd"],
    labels=["Groundwater", "Surface water"],
    colors=["#8c6d31", "#1f77b4"],
    alpha=0.85,
)
ax.set_title("Source of Wisconsin public-supply withdrawals, 2020")
ax.set_ylabel("Withdrawal (million gallons/day)")
ax.set_xlabel("Month")
ax.legend(loc="upper left")
fig.tight_layout()

## Notes

- **Spatial resolution.** Estimates are always returned on the HUC12 grid,
  regardless of the area you query. Joining `huc12_id` to a HUC12
  boundary layer (for example with `geopandas`) lets you map the results.
- **Pick one selector.** Use exactly one of `state`, `county` (5-digit FIPS),
  or `huc`. `state` accepts a name, postal code, or FIPS; a `huc` code's length
  sets its level (`huc="04"` → HUC2, `huc="07070005"` → HUC8). Each selector
  also accepts a list (e.g. `state=["WI", "MN"]`), fanned out into one request
  per area and concatenated.
- **Large areas paginate.** A query spanning more than `limit` HUC12s (600 by
  default) is split across pages; `get_wateruse` follows the links and returns
  a single concatenated frame. Whole-region queries (e.g. `huc="04"`)
  may return many pages and take longer.
- **Annual data.** Set `time_resolution="annualcy"` (calendar year) or `"annualwy"`
  (water year) and use four-digit years for `start_date`/`end_date`; the time
  column comes back as `year` instead of `year_month`.
- **Other models.** Swap `model` and `variable` to retrieve irrigation
  (`wu-irrigation-wd`) or thermoelectric (`wu-thermoelectric`) water use. See
  the [NWDC data catalog](https://water.usgs.gov/nwaa-data/) for the variables
  available in each.